# 🌱 PlantGuard AI — Plant Disease Detection System

An end-to-end Machine Learning pipeline utilizing **EfficientNetB0 transfer learning** trained on the real **PlantVillage dataset** (54,305 images across 38 classes) to diagnose crop foliage diseases in real time.

In [1]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

# Add project root to sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.config import Config, default_config
from ml.predict import predict_leaf

print(f'TensorFlow Version: {tf.__version__}')
print(f'Project Root: {PROJECT_ROOT}')

## 1. Load Classes and Evaluation Metrics
We load the class mapping containing all 38 supported disease categories, along with verified test metrics evaluated on 1,124 held-out real test images.

In [2]:
class_path = PROJECT_ROOT / 'models' / 'class_names.json'
metrics_path = PROJECT_ROOT / 'results' / 'metrics.json'

with open(class_path, 'r') as f:
    class_names = json.load(f)

with open(metrics_path, 'r') as f:
    metrics = json.load(f)

print(f'Total Classes Supported: {len(class_names)}')
print(f'Test Accuracy: {metrics["accuracy_percentage"]}%')
print(f'Macro Precision: {metrics["precision_macro"] * 100:.2f}%')
print(f'Macro Recall:    {metrics["recall_macro"] * 100:.2f}%')
print(f'Macro F1-Score:  {metrics["f1_macro"] * 100:.2f}%')

## 2. Visualize Model Evaluation Artifacts
The 38x38 confusion matrix and training curves show strong diagonal dominance and rapid convergence.

In [3]:
cm_img_path = PROJECT_ROOT / 'results' / 'confusion_matrix.png'
if cm_img_path.exists():
    cm_img = Image.open(cm_img_path)
    plt.figure(figsize=(10, 10))
    plt.imshow(cm_img)
    plt.axis('off')
    plt.title('PlantGuard AI — 38-Class Confusion Matrix', fontsize=14)
    plt.show()

## 3. End-to-End Prediction on Real Leaf Specimens
Test predictions on both healthy and diseased specimens from our dataset sample gallery.

In [4]:
sample_img = PROJECT_ROOT / 'dataset' / 'samples' / 'Tomato_Early_Blight.jpg'
result = predict_leaf(sample_img, top_k=3)

print('DIAGNOSTIC REPORT:')
print(f'Plant:       {result["plant"]}')
print(f'Diagnosis:   {result["disease"]}')
print(f'Confidence:  {result["confidence"]} %')
print(f'Status:      {result["status"]}')
print(f'Severity:    {result["severity"]}')
print('\nTop-3 Predictions:')
for p in result['top_predictions']:
    print(f'  - {p["plant"]} ({p["disease"]}) : {p["confidence"]} %')